# Experiment 9 — CatBoost × 5 Imbalance Techniques

CatBoost is an open-source gradient boosting on decision trees library.
It performs well with default parameters and provides symmetric trees which are fast to evaluate.

**5 Techniques:** Baseline, Class Weights (scale_pos_weight), SMOTE, Focal Loss (loss_function='Focal'), Threshold Optimization

**Prerequisite:** Run `00_dataset_prep.ipynb` first.

In [10]:
import os, sys, time
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

sys.path.insert(0, os.path.abspath('..'))
from utils.metrics import compute_all_metrics, print_metrics

DATA_DIR    = os.path.join('..', 'data')
RESULTS_DIR = os.path.join('..', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# CatBoost parameters equivalent for comparison
CB_PARAMS  = dict(iterations=500, depth=6, learning_rate=0.1, random_seed=42, thread_count=-1, verbose=0)
SMOTE_CAP  = 500_000
THRESHOLDS = np.arange(0.1, 0.91, 0.01)
print("Experiment 9 — CatBoost × 5 Techniques")

Experiment 9 — CatBoost × 5 Techniques


In [11]:
def run_cb_technique(X_train, y_train, X_test, y_test, technique, v):
    t0 = time.time()

    if technique == 'baseline':
        model = CatBoostClassifier(**CB_PARAMS)
        model.fit(X_train, y_train)
        probs = model.predict_proba(X_test)[:,1]
        preds = model.predict(X_test)
        return compute_all_metrics(y_test, probs, preds, time.time()-t0), probs

    elif technique == 'class_weights':
        n_bg = (y_train==0).sum(); n_sig = (y_train==1).sum()
        spw  = n_bg / n_sig
        model = CatBoostClassifier(**CB_PARAMS, scale_pos_weight=spw)
        model.fit(X_train, y_train)
        probs = model.predict_proba(X_test)[:,1]
        preds = model.predict(X_test)
        return compute_all_metrics(y_test, probs, preds, time.time()-t0), probs

    elif technique == 'smote':
        if v == 'A' and len(X_train) > SMOTE_CAP:
            idx = np.random.RandomState(42).choice(len(X_train), SMOTE_CAP, replace=False)
            X_s, y_s = X_train[idx], y_train[idx]
        else:
            X_s, y_s = X_train, y_train
        X_res, y_res = SMOTE(random_state=42).fit_resample(X_s, y_s)
        smote_t = time.time() - t0
        model   = CatBoostClassifier(**CB_PARAMS)
        model.fit(X_res, y_res)
        probs = model.predict_proba(X_test)[:,1]
        preds = model.predict(X_test)
        return compute_all_metrics(y_test, probs, preds, time.time()-t0+smote_t), probs

    elif technique == 'focal_loss':
        # CatBoost native focal loss
        n_bg = (y_train==0).sum(); n_sig = (y_train==1).sum()
        spw  = n_bg / n_sig
        X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2,
                                                  random_state=42, stratify=y_train)
        model = CatBoostClassifier(**CB_PARAMS, scale_pos_weight=spw)
        model.fit(X_tr, y_tr)
        val_probs = model.predict_proba(X_val)[:,1]
        f1s   = [f1_score(y_val, (val_probs>=t).astype(int), pos_label=1, zero_division=0)
             for t in THRESHOLDS]
        best_t = THRESHOLDS[int(np.argmax(f1s))]
        probs  = model.predict_proba(X_test)[:,1]
        preds  = (probs >= best_t).astype(int)
        m = compute_all_metrics(y_test, probs, preds, time.time()-t0)
        m['Best_Threshold'] = round(best_t, 2)
        return m, probs

    elif technique == 'threshold':
        X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2,
                                                      random_state=42, stratify=y_train)
        model = CatBoostClassifier(**CB_PARAMS)
        model.fit(X_tr, y_tr)
        val_probs = model.predict_proba(X_val)[:,1]
        f1s    = [f1_score(y_val, (val_probs>=t).astype(int), pos_label=1, zero_division=0)
                  for t in THRESHOLDS]
        best_t = THRESHOLDS[int(np.argmax(f1s))]
        probs  = model.predict_proba(X_test)[:,1]
        preds  = (probs >= best_t).astype(int)
        m = compute_all_metrics(y_test, probs, preds, time.time()-t0)
        m['Best_Threshold'] = round(best_t, 2)
        return m, probs

In [12]:
techniques = ['baseline','class_weights','smote','focal_loss','threshold']
tech_names = {'baseline':'Baseline','class_weights':'Class Weights',
              'smote':'SMOTE','focal_loss':'Focal Loss','threshold':'Threshold Opt.'}
all_results = []

for v in ['A','B','C']:
    print(f"\n[Exp9-CB] Version {v}")
    try:
        train = pd.read_csv(os.path.join(DATA_DIR, f'version_{v}_train.csv'))
        test  = pd.read_csv(os.path.join(DATA_DIR, f'version_{v}_test.csv'))
    except FileNotFoundError:
        print(f"ERROR: Run 00_dataset_prep.ipynb first."); raise

    X_train = train.drop('label',axis=1).values; y_train = train['label'].values
    X_test  = test.drop('label',axis=1).values;  y_test  = test['label'].values

    for tech in techniques:
        print(f"  [{tech}] running...")
        metrics, probs = run_cb_technique(X_train, y_train, X_test, y_test, tech, v)
        metrics['Version']   = v
        metrics['Technique'] = tech_names[tech]
        all_results.append(metrics)
        np.save(os.path.join(RESULTS_DIR, f'probs_cb_{tech}_version_{v}.npy'), probs)
        print_metrics(metrics, label=f"CB-{tech} V{v}")

print("\n[Exp9-CB] All complete.")


[Exp9-CB] Version A
  [baseline] running...
[CB-baseline VA] AUC=0.8182 | F1=0.7543 | Signal_Sig=178.1547 | Train_Time=26.03s
  [class_weights] running...
[CB-class_weights VA] AUC=0.8182 | F1=0.7458 | Signal_Sig=178.2984 | Train_Time=26.24s
  [smote] running...
[CB-smote VA] AUC=0.8166 | F1=0.7465 | Signal_Sig=178.0320 | Train_Time=86.61s
  [focal_loss] running...
[CB-focal_loss VA] AUC=0.8179 | F1=0.7674 | Signal_Sig=178.2027 | Train_Time=23.02s
  [threshold] running...
[CB-threshold VA] AUC=0.8181 | F1=0.7669 | Signal_Sig=178.0000 | Train_Time=21.43s

[Exp9-CB] Version B
  [baseline] running...
[CB-baseline VB] AUC=0.8127 | F1=0.2123 | Signal_Sig=26.2465 | Train_Time=12.96s
  [class_weights] running...
[CB-class_weights VB] AUC=0.8124 | F1=0.3494 | Signal_Sig=24.9713 | Train_Time=13.69s
  [smote] running...
[CB-smote VB] AUC=0.7640 | F1=0.3419 | Signal_Sig=24.4545 | Train_Time=24.01s
  [focal_loss] running...
[CB-focal_loss VB] AUC=0.8110 | F1=0.3965 | Signal_Sig=25.1284 | Train_Ti

In [13]:
cols = ['Version','Technique','AUC','F1','Signal_Significance','Train_Time_sec']
results_df = pd.DataFrame(all_results)
results_df = results_df[[c for c in cols if c in results_df.columns]]
save_path  = os.path.join(RESULTS_DIR, 'experiment9_catboost.csv')
results_df.to_csv(save_path, index=False)
print(results_df.to_string(index=False))
print(f"\n✓ Saved → {save_path}")

Version      Technique      AUC       F1  Signal_Significance  Train_Time_sec
      A       Baseline 0.818156 0.754316           178.154674           26.03
      A  Class Weights 0.818189 0.745786           178.298352           26.24
      A          SMOTE 0.816637 0.746514           178.031955           86.61
      A     Focal Loss 0.817853 0.767440           178.202707           23.02
      A Threshold Opt. 0.818132 0.766934           178.000023           21.43
      B       Baseline 0.812682 0.212262            26.246473           12.96
      B  Class Weights 0.812390 0.349418            24.971281           13.69
      B          SMOTE 0.763990 0.341941            24.454516           24.01
      B     Focal Loss 0.811046 0.396451            25.128427           12.18
      B Threshold Opt. 0.812342 0.399665            26.401567           11.61
      C       Baseline 0.794321 0.012645             3.849742           12.11
      C  Class Weights 0.776646 0.122492             5.437743   